# Image Caption Generator — Exploratory Notebook

This notebook walks through the complete pipeline:
1. Dataset exploration
2. Preprocessing
3. Feature extraction
4. Model training
5. Evaluation and prediction

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Dataset Exploration

In [ ]:
from src.preprocessing import load_captions, clean_caption
from src.config import CAPTIONS_FILE, IMAGES_DIR

captions = load_captions()
print(f"Total images: {len(captions)}")
print(f"Total captions: {sum(len(v) for v in captions.values())}")

# Sample captions
sample_image = list(captions.keys())[0]
print(f"\nSample image: {sample_image}")
for i, cap in enumerate(captions[sample_image]):
    print(f"  Caption {i+1}: {cap}")

In [ ]:
# Caption length distribution
lengths = [len(cap.split()) for caps in captions.values() for cap in caps]
plt.figure(figsize=(8, 4))
plt.hist(lengths, bins=30, color='#667eea', edgecolor='white')
plt.xlabel('Caption Length (tokens)')
plt.ylabel('Frequency')
plt.title('Caption Length Distribution')
plt.show()
print(f"Mean length: {np.mean(lengths):.1f} | Max: {max(lengths)}")

## 2. Preprocessing & Vocabulary

In [ ]:
from src.preprocessing import run_preprocessing, build_vocabulary

train, val, test, word2idx, idx2word, vocab_size = run_preprocessing()
print(f"Vocabulary size: {vocab_size}")
print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")

## 3. VGG16 Feature Extraction

In [ ]:
from src.feature_extraction import extract_features_for_dataset, VGG16FeatureExtractor
from PIL import Image

# Extract features for a small subset (for demo)
sample_images = list(train.keys())[:10]
features = extract_features_for_dataset(sample_images)
print(f"Feature shape: {next(iter(features.values())).shape}")

## 4. Model Training

In [ ]:
# Run full training (uncomment to execute)
# from src.train import train_model
# history = train_model(epochs=10, batch_size=32)
# print("Training complete!")

## 5. Prediction & Evaluation

In [ ]:
from src.predict import CaptionGenerator
from src.config import IMAGES_DIR

try:
    generator = CaptionGenerator()
    test_image = IMAGES_DIR / list(test.keys())[0]
    result = generator.predict_from_path(str(test_image))
    print(f"Image: {test_image.name}")
    print(f"Generated: {result['caption']}")
    print(f"Confidence: {result['confidence']:.2%}")
    print(f"Reference: {test[list(test.keys())[0]][0]}")
except FileNotFoundError as e:
    print(f"Model not trained yet: {e}")

In [ ]:
# Plot training history if available
import json
from src.config import TRAINING_HISTORY_PATH
from src.evaluation import plot_training_history

if TRAINING_HISTORY_PATH.exists():
    with open(TRAINING_HISTORY_PATH) as f:
        history = json.load(f)
    plot_training_history(history)
    from IPython.display import Image as IPImage
    display(IPImage(filename='../outputs/plots/training_history.png'))